# Training a real network, tracked in Weights & Biases

Before you start: `uv add pillow`, and you're logged in to W&B (`uv run wandb login`).

Every training run in this notebook logs to the W&B project `ml-lab-fashion`. Each run prints a link. Keep the project page open in a browser tab and compare runs there as you go.

In [ ]:
import math
import os
import time

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
import matplotlib.pyplot as plt
import wandb
from datasets import load_dataset

if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")
print("Device:", device)

def set_seed(seed):
    torch.manual_seed(seed)
    np.random.seed(seed)

set_seed(0)
PROJECT = "ml-lab-fashion"

## 1. The data

In [ ]:
ds = load_dataset("zalando-datasets/fashion_mnist")
print(ds)
CLASSES = ds["train"].features["label"].names
print(CLASSES)

The images come back as PIL images. Convert everything to tensors once, scaled to the range 0 to 1. The whole training set is about 190 MB as float32, so it fits in memory easily.

In [ ]:
def to_tensors(split):
    images = np.stack([np.array(img, dtype=np.uint8) for img in split["image"]])
    X = torch.from_numpy(images).float() / 255.0
    y = torch.tensor(list(split["label"]))
    return X, y

start = time.time()
X_full, y_full = to_tensors(ds["train"])
X_test, y_test = to_tensors(ds["test"])
print(f"train {tuple(X_full.shape)}  test {tuple(X_test.shape)}  ({time.time() - start:.1f}s)")

In [ ]:
fig, axes = plt.subplots(2, 8, figsize=(14, 4))
for ax, i in zip(axes.flat, range(16)):
    ax.imshow(X_full[i], cmap="gray")
    ax.set_title(CLASSES[y_full[i]], fontsize=9)
    ax.axis("off")
plt.show()

## 2. Three splits

- Training set: the model learns from it.
- Validation set: you look at it to make decisions (architecture, learning rate, when to stop). Carved out of the original training data.
- Test set: touched once, at the very end, to get an honest estimate of real-world accuracy. If you choose models by test score, the test score stops being honest, because you've indirectly fit it.

Normalization uses statistics from the training set only. Using validation or test statistics would leak information about them into training.

In [ ]:
g = torch.Generator().manual_seed(0)
perm = torch.randperm(len(X_full), generator=g)
n_val = len(X_full) // 6                     # 10,000 of the 60,000
val_idx, train_idx = perm[:n_val], perm[n_val:]

X_train, y_train = X_full[train_idx], y_full[train_idx]
X_val, y_val = X_full[val_idx], y_full[val_idx]

mean, std = X_train.mean(), X_train.std()
X_train = (X_train - mean) / std
X_val = (X_val - mean) / std
X_test = (X_test - mean) / std

print(f"train {len(X_train)}  val {len(X_val)}  test {len(X_test)}")
print("examples per class in train:", torch.bincount(y_train).tolist())

A `DataLoader` serves the data in shuffled mini-batches. Each optimizer step uses one batch, and one pass over all batches is an epoch.

In [ ]:
def make_loader(X, y, batch_size, shuffle):
    return DataLoader(TensorDataset(X, y), batch_size=batch_size, shuffle=shuffle)

train_loader = make_loader(X_train, y_train, 128, shuffle=True)
val_loader = make_loader(X_val, y_val, 512, shuffle=False)
test_loader = make_loader(X_test, y_test, 512, shuffle=False)

xb, yb = next(iter(train_loader))
print("batch:", tuple(xb.shape), tuple(yb.shape), "| steps per epoch:", len(train_loader))

## 3. Cross-entropy, and the step-0 sanity check

A classifier outputs one logit per class. Softmax turns logits into probabilities, and cross-entropy is the negative log of the probability assigned to the correct class. Confident and right gives a loss near 0. Confident and wrong gives a big loss.

In [ ]:
logits = torch.tensor([[2.0, 0.5, -1.0]])
target = torch.tensor([0])

probs = torch.softmax(logits, dim=-1)
print("probabilities:", probs)
print("by hand:      ", -torch.log(probs[0, target]).item())
print("F.cross_entropy:", F.cross_entropy(logits, target).item())

In [ ]:
class MLP(nn.Module):
    def __init__(self, hidden=256, dropout=0.0):
        super().__init__()
        self.net = nn.Sequential(
            nn.Flatten(),                       # (N, 28, 28) -> (N, 784)
            nn.Linear(784, hidden), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(hidden, hidden), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(hidden, 10),
        )

    def forward(self, x):
        return self.net(x)

model = MLP().to(device)
with torch.no_grad():
    loss0 = F.cross_entropy(model(xb.to(device)), yb.to(device))
print(f"loss at step 0: {loss0.item():.3f}   expected for 10 classes: -ln(1/10) = {math.log(10):.3f}")

A freshly initialized network should be roughly uniform across the classes, so its loss should be close to ln(10) ≈ 2.30. If your step-0 loss is far off (say 15), something is wrong with initialization or the scale of the inputs, and you've found it before wasting a training run. Check this on every new model.

## 4. The training loop, with evaluation and logging

This `train` function is what you'll reuse for the rest of the notebook. It adds four things to the loop from module 02:

1. A validation pass after every epoch, with the model in eval mode (dropout off) and gradients off.
2. A learning-rate schedule (used in section 7).
3. Accuracy tracking alongside loss.
4. Logging to W&B.

In [ ]:
@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    total_loss, correct, n = 0.0, 0, 0
    for X, y in loader:
        X, y = X.to(device), y.to(device)
        logits = model(X)
        total_loss += F.cross_entropy(logits, y, reduction="sum").item()
        correct += (logits.argmax(dim=1) == y).sum().item()
        n += len(y)
    model.train()
    return total_loss / n, correct / n


def lr_at(step, total_steps, base_lr, warmup_steps, schedule):
    if schedule == "constant":
        return base_lr
    if step < warmup_steps:                                  # linear warmup
        return base_lr * (step + 1) / warmup_steps
    progress = (step - warmup_steps) / max(1, total_steps - warmup_steps)
    return base_lr * 0.5 * (1 + math.cos(math.pi * progress))  # cosine decay to 0


def train(model, config, train_loader, val_loader, run_name=None, run=None, augment=None):
    cfg = {"epochs": 5, "lr": 1e-3, "weight_decay": 0.0, "schedule": "constant", "warmup_frac": 0.05, **config}
    if run is None:
        run = wandb.init(project=PROJECT, name=run_name, config=cfg)
    else:
        run.config.update(cfg, allow_val_change=True)

    model.to(device).train()
    opt = torch.optim.AdamW(model.parameters(), lr=cfg["lr"], weight_decay=cfg["weight_decay"])
    total_steps = cfg["epochs"] * len(train_loader)
    warmup_steps = max(1, int(cfg["warmup_frac"] * total_steps))
    history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}
    step, start = 0, time.time()

    for epoch in range(cfg["epochs"]):
        loss_sum, correct, seen = 0.0, 0, 0
        for X, y in train_loader:
            X, y = X.to(device), y.to(device)
            if augment is not None:
                X = augment(X)
            lr = lr_at(step, total_steps, cfg["lr"], warmup_steps, cfg["schedule"])
            for group in opt.param_groups:
                group["lr"] = lr

            logits = model(X)
            loss = F.cross_entropy(logits, y)
            opt.zero_grad()
            loss.backward()
            opt.step()

            loss_sum += loss.item() * len(y)
            correct += (logits.argmax(dim=1) == y).sum().item()
            seen += len(y)
            if step % 50 == 0:
                run.log({"train/loss_step": loss.item(), "lr": lr}, step=step)
            step += 1

        val_loss, val_acc = evaluate(model, val_loader)
        history["train_loss"].append(loss_sum / seen)
        history["train_acc"].append(correct / seen)
        history["val_loss"].append(val_loss)
        history["val_acc"].append(val_acc)
        run.log({"epoch": epoch + 1, "train/loss": loss_sum / seen, "train/acc": correct / seen,
                 "val/loss": val_loss, "val/acc": val_acc}, step=step)
        if cfg["epochs"] <= 10 or (epoch + 1) % 10 == 0:
            print(f"epoch {epoch + 1:3}  train loss {loss_sum / seen:.4f} acc {correct / seen:.3f} | "
                  f"val loss {val_loss:.4f} acc {val_acc:.3f}  ({time.time() - start:.0f}s)")

    run.summary["best_val_acc"] = max(history["val_acc"])
    run.finish()
    return history

Train the MLP. Watch the printed W&B link and open it.

In [ ]:
set_seed(0)
mlp = MLP(hidden=256)
print(f"MLP parameters: {sum(p.numel() for p in mlp.parameters()):,}")
hist_mlp = train(mlp, {"epochs": 5, "lr": 1e-3, "model": "mlp"}, train_loader, val_loader, run_name="mlp-baseline")

Expect around 88% validation accuracy. One thing to notice: the reported train accuracy is measured while dropout is active and while the weights are still changing during the epoch, so it isn't exactly comparable to the validation number.

## 5. A convolutional network

The MLP flattens each image into 784 unrelated numbers, so it has to learn separately that a sleeve on the left and a sleeve on the right are the same thing. A convolution slides one small filter (here 3x3) across the whole image, reusing the same weights everywhere. That builds in two facts about images: nearby pixels are related, and a pattern means the same thing wherever it appears. Pooling then shrinks the image so later layers see larger regions.

In [ ]:
class CNN(nn.Module):
    def __init__(self, dropout=0.25, batchnorm=False):
        super().__init__()

        def block(c_in, c_out):
            layers = [nn.Conv2d(c_in, c_out, kernel_size=3, padding=1)]
            if batchnorm:
                layers.append(nn.BatchNorm2d(c_out))
            layers += [nn.ReLU(), nn.MaxPool2d(2)]
            return layers

        self.features = nn.Sequential(*block(1, 32), *block(32, 64))
        self.head = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 7 * 7, 128), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(128, 10),
        )

    def forward(self, x):
        if x.dim() == 3:
            x = x.unsqueeze(1)   # (N, 28, 28) -> (N, 1, 28, 28): conv layers expect a channel dimension
        return self.head(self.features(x))


# Trace the shapes through every layer. Predict each one first.
x = torch.zeros(1, 1, 28, 28)
cnn = CNN()
for layer in list(cnn.features) + list(cnn.head):
    x = layer(x)
    print(f"{layer.__class__.__name__:10} -> {tuple(x.shape)}")
print(f"CNN parameters: {sum(p.numel() for p in cnn.parameters()):,}")

Notice where the parameters live. The two conv layers have under 20,000 weights between them, and the first linear layer has about 400,000. Convolutions are cheap in parameters but expensive in compute, because each filter runs at every position.

In [ ]:
set_seed(0)
cnn = CNN()
hist_cnn = train(cnn, {"epochs": 5, "lr": 1e-3, "model": "cnn"}, train_loader, val_loader, run_name="cnn-baseline")

In [ ]:
plt.plot(range(1, 6), hist_mlp["val_acc"], "o-", label="MLP")
plt.plot(range(1, 6), hist_cnn["val_acc"], "o-", label="CNN")
plt.xlabel("epoch"); plt.ylabel("validation accuracy"); plt.legend(); plt.title("MLP vs CNN")
plt.show()

Expect the CNN to land around 91%. In W&B, go to the project page and look at `val/acc` with both runs selected.

## 6. Overfitting

Give a big model very little data and it memorizes the training set instead of learning patterns that generalize. Here: a wide MLP, only 1,000 training images, 60 epochs, no regularization.

In [ ]:
small_idx = torch.randperm(len(X_train), generator=torch.Generator().manual_seed(1))[:1000]
small_loader = make_loader(X_train[small_idx], y_train[small_idx], 64, shuffle=True)

set_seed(0)
hist_overfit = train(MLP(hidden=512, dropout=0.0),
                     {"epochs": 60, "lr": 1e-3, "model": "mlp-512", "train_size": 1000},
                     small_loader, val_loader, run_name="small-data-no-reg")

Now the same model with the two standard fixes:

- Dropout randomly zeroes half the activations during training, so the network can't rely on any single path through it.
- Weight decay (in AdamW) shrinks every weight a little each step, which favors simpler functions.

In [ ]:
set_seed(0)
hist_reg = train(MLP(hidden=512, dropout=0.5),
                 {"epochs": 60, "lr": 1e-3, "weight_decay": 0.1, "dropout": 0.5, "model": "mlp-512", "train_size": 1000},
                 small_loader, val_loader, run_name="small-data-dropout-wd")

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
for hist, name in [(hist_overfit, "no regularization"), (hist_reg, "dropout + weight decay")]:
    ax1.plot(hist["train_acc"], label=f"train, {name}")
    ax1.plot(hist["val_acc"], "--", label=f"val, {name}")
    ax2.plot(hist["val_loss"], label=name)
ax1.set_title("accuracy"); ax1.set_xlabel("epoch"); ax1.legend(fontsize=8)
ax2.set_title("validation loss"); ax2.set_xlabel("epoch"); ax2.legend(fontsize=8)
plt.show()

What to read off these curves:

- Without regularization, training accuracy reaches 100% while validation accuracy stalls. The gap between them is overfitting.
- Validation loss falls, then climbs, even while validation accuracy holds steady. The model is getting more confident on the examples it gets wrong. In a real project, you'd stop training (or keep the checkpoint) where validation loss bottomed out. That's early stopping.
- With dropout and weight decay, the gap shrinks.

The best fix for overfitting is more data. With all 50,000 images, the same model barely overfits in 5 epochs.

## 7. Learning-rate schedules

A fixed learning rate is a compromise: big enough to make progress early, small enough to settle at the end. Schedules do both. Warmup ramps the rate up over the first few percent of steps (large updates at the very start, when the weights are random, can knock training off course), then cosine decay lowers it smoothly to zero. Almost every LLM is trained with warmup plus cosine or something close to it.

In [ ]:
total = 5 * len(train_loader)
lrs = [lr_at(s, total, 3e-3, int(0.05 * total), "cosine") for s in range(total)]
plt.plot(lrs); plt.xlabel("step"); plt.ylabel("learning rate"); plt.title("Warmup + cosine decay")
plt.show()

In [ ]:
set_seed(0)
cnn_cos = CNN()
hist_cos = train(cnn_cos, {"epochs": 5, "lr": 3e-3, "schedule": "cosine", "model": "cnn"},
                 train_loader, val_loader, run_name="cnn-cosine")

Compare `cnn-cosine` against `cnn-baseline` in W&B, and look at the `lr` chart for both runs. The cosine run uses a higher peak rate and usually finishes a bit better, because the low rate at the end lets it settle into a minimum.

## 8. Checkpoints, and the one-time test

A checkpoint saves the weights (`state_dict`) plus whatever you need to rebuild the model later. Checkpoints go in `checkpoints/`, which git ignores.

In [ ]:
os.makedirs("checkpoints", exist_ok=True)
path = "checkpoints/cnn_cosine.pt"
torch.save({
    "model_state": cnn_cos.state_dict(),
    "model_args": {"dropout": 0.25, "batchnorm": False},
    "classes": CLASSES,
    "normalization": {"mean": mean.item(), "std": std.item()},
}, path)
print(f"saved {os.path.getsize(path) / 1e6:.1f} MB")

ckpt = torch.load(path, map_location=device)
restored = CNN(**ckpt["model_args"]).to(device)
restored.load_state_dict(ckpt["model_state"])
print("original val:", evaluate(cnn_cos, val_loader))
print("restored val:", evaluate(restored, val_loader))

Same numbers, so the checkpoint is complete. Note that the normalization stats are saved too: any new image has to be normalized exactly as the training data was, or predictions quietly degrade.

You've made your choices using validation data. Now, once, check the test set.

In [ ]:
test_loss, test_acc = evaluate(restored, test_loader)
print(f"test accuracy: {test_acc:.3f}")

restored.eval()
with torch.no_grad():
    preds = restored(X_test[:16].to(device)).argmax(dim=1).cpu()

fig, axes = plt.subplots(2, 8, figsize=(14, 4))
for ax, img, p, t in zip(axes.flat, X_test[:16], preds, y_test[:16]):
    ax.imshow(img, cmap="gray")
    ax.set_title(f"{CLASSES[p]}\n(true: {CLASSES[t]})", fontsize=7, color="green" if p == t else "red")
    ax.axis("off")
plt.show()

## 9. A hyperparameter sweep (optional, about 10 minutes)

A sweep runs many trainings with different settings and lets W&B chart which settings worked. Random search over a sensible range is usually a better use of a fixed budget than a grid. Set `RUN_SWEEP = True` to run 8 short trainings, then open the sweep page W&B prints and look at the parallel coordinates chart.

In [ ]:
RUN_SWEEP = False

sweep_config = {
    "method": "random",
    "metric": {"name": "val/acc", "goal": "maximize"},
    "parameters": {
        "lr": {"distribution": "log_uniform_values", "min": 3e-4, "max": 1e-2},
        "dropout": {"values": [0.0, 0.25, 0.5]},
        "batch_size": {"values": [64, 128, 256]},
    },
}

def sweep_run():
    run = wandb.init()
    cfg = dict(run.config)
    set_seed(0)
    loader = make_loader(X_train, y_train, cfg["batch_size"], shuffle=True)
    model = CNN(dropout=cfg["dropout"])
    train(model, {"epochs": 2, "lr": cfg["lr"], "schedule": "cosine", "model": "cnn"}, loader, val_loader, run=run)

if RUN_SWEEP:
    sweep_id = wandb.sweep(sweep_config, project=PROJECT)
    wandb.agent(sweep_id, function=sweep_run, count=8)

---
# Exercises

**Exercise 1: Confusion matrix.** For `cnn_cos` on the validation set, build the 10x10 confusion matrix (row = true class, column = predicted) without loops. Hint: `torch.bincount(true * 10 + pred, minlength=100)`. Plot it with `plt.imshow` and print the five most common mistakes. Which classes does the model confuse, and does that make sense to you as a human looking at the images?

**Exercise 2: Overfit one batch.** Take a single batch of 32 training images and train a fresh `CNN(dropout=0.0)` on only that batch for 200 steps. The loss should go to nearly zero. This is the first debugging check for any new model or training loop: if a model can't memorize 32 examples, there's a bug (wrong labels, a missing `zero_grad`, a broken loss), and no amount of tuning will fix it.

**Exercise 3: Learning-rate range test.** Start a fresh CNN at learning rate 1e-6 and multiply the rate by a constant factor every step so it reaches 1.0 after 200 steps. Record the loss at every step, stop early if the loss exceeds 4 times its minimum, and plot loss against learning rate on a log x-axis. A good learning rate is usually about 10 times smaller than the one where loss is lowest.

**Exercise 4: BatchNorm.** Train `CNN(batchnorm=True)` with the same settings as `cnn-cosine` and compare in W&B. BatchNorm normalizes each channel's activations within a batch, which usually makes training faster and less sensitive to the learning rate. Did it here?

**Exercise 5: Data augmentation.** On the 1,000-image `small_loader`, train a `CNN` for 40 epochs twice: once plain, and once with an `augment` function passed to `train` that randomly flips half of each batch left to right and shifts the whole batch by up to 2 pixels (`torch.roll`). Augmentation creates new training examples for free. Does validation accuracy improve?

---
# Solutions

In [ ]:
# Exercise 1
cnn_cos.eval()
preds, trues = [], []
with torch.no_grad():
    for X, y in val_loader:
        preds.append(cnn_cos(X.to(device)).argmax(dim=1).cpu())
        trues.append(y)
preds, trues = torch.cat(preds), torch.cat(trues)

cm = torch.bincount(trues * 10 + preds, minlength=100).reshape(10, 10)

plt.figure(figsize=(7, 6))
plt.imshow(cm, cmap="Blues")
plt.xticks(range(10), CLASSES, rotation=60, ha="right", fontsize=8)
plt.yticks(range(10), CLASSES, fontsize=8)
plt.xlabel("predicted"); plt.ylabel("true"); plt.colorbar()
plt.show()

mistakes = cm.clone()
mistakes.fill_diagonal_(0)
values, flat_idx = mistakes.flatten().topk(5)
for v, i in zip(values, flat_idx):
    print(f"{v.item():4} times: true {CLASSES[i // 10]!r} predicted as {CLASSES[i % 10]!r}")

In [ ]:
# Exercise 2
set_seed(0)
m = CNN(dropout=0.0).to(device)
opt = torch.optim.AdamW(m.parameters(), lr=1e-3)
xb, yb = next(iter(train_loader))
xb, yb = xb[:32].to(device), yb[:32].to(device)

for step in range(200):
    loss = F.cross_entropy(m(xb), yb)
    opt.zero_grad(); loss.backward(); opt.step()
    if step % 40 == 0 or step == 199:
        print(f"step {step:3}  loss {loss.item():.5f}")

In [ ]:
# Exercise 3
set_seed(0)
m = CNN().to(device)
opt = torch.optim.AdamW(m.parameters(), lr=1e-6)
lr, factor = 1e-6, (1.0 / 1e-6) ** (1 / 200)
lrs, losses = [], []
batches = iter(train_loader)

for step in range(200):
    try:
        X, y = next(batches)
    except StopIteration:
        batches = iter(train_loader)
        X, y = next(batches)
    for group in opt.param_groups:
        group["lr"] = lr
    loss = F.cross_entropy(m(X.to(device)), y.to(device))
    opt.zero_grad(); loss.backward(); opt.step()
    lrs.append(lr); losses.append(loss.item())
    if loss.item() > 4 * min(losses):
        break
    lr *= factor

best = lrs[int(np.argmin(losses))]
plt.semilogx(lrs, losses)
plt.xlabel("learning rate"); plt.ylabel("loss"); plt.title(f"LR range test (lowest loss at {best:.1e})")
plt.show()
print(f"suggested learning rate: about {best / 10:.1e}")

In [ ]:
# Exercise 4
set_seed(0)
hist_bn = train(CNN(batchnorm=True), {"epochs": 5, "lr": 3e-3, "schedule": "cosine", "model": "cnn-bn"},
                train_loader, val_loader, run_name="cnn-cosine-batchnorm")
print(f"best val acc  cosine: {max(hist_cos['val_acc']):.3f}   cosine + batchnorm: {max(hist_bn['val_acc']):.3f}")

In [ ]:
# Exercise 5
def augment(X):
    flip = torch.rand(len(X), device=X.device) < 0.5
    X = torch.where(flip[:, None, None], X.flip(-1), X)          # mirror half the batch left-right
    dx, dy = np.random.randint(-2, 3, size=2)
    return torch.roll(X, shifts=(int(dy), int(dx)), dims=(-2, -1))   # shift the batch a little

set_seed(0)
hist_plain = train(CNN(), {"epochs": 40, "lr": 1e-3, "model": "cnn", "train_size": 1000},
                   small_loader, val_loader, run_name="small-data-cnn")
set_seed(0)
hist_aug = train(CNN(), {"epochs": 40, "lr": 1e-3, "model": "cnn", "train_size": 1000, "augment": True},
                 small_loader, val_loader, run_name="small-data-cnn-augmented", augment=augment)
print(f"best val acc  plain: {max(hist_plain['val_acc']):.3f}   augmented: {max(hist_aug['val_acc']):.3f}")